# Sidechain — 2025 dry run

Rehearse the full loop on the 2025 VCC data before the Aug-20 2026 drop:

    load split → normalize → fit rung → predict held-out perturbations → score on the local cell-eval mirror

**Read this before quoting any number below.** Arc never released the 2025
public/private test AnnData to entrants. All we hold is `adata_Training.h5ad`,
a locally derived subset, `gene_names.csv` and `pert_counts_Validation.csv`
(50 rows of perturbation *counts*, no expression). The holdout here is therefore
carved out of the **training set**. Numbers from this notebook are a pipeline
check — never "what we would have placed in 2025".

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("cell_eval").setLevel(logging.WARNING)

from sidechain.data import loaders
from sidechain.data.pseudobulk import pseudobulk, delta_vs_control
from sidechain.data.registry import load_registry
from sidechain.eval.local_mirror import score, summarize
from sidechain.models.baseline_stats import (
    PredictControl, PredictMeanPerturbation, StatisticalBackbone,
)

cfg = loaders.load_challenge_config("../challenges/vcc2025/config.yaml")
PERT_COL, CONTROL = cfg["pert_col"], cfg["control_label"]
cfg

## 1. Load

`dev=True` uses `adata_Training_subset.h5ad` (3.1 GB) rather than the 14 GB full
file. That subset was derived locally on 2025-11-23, exists in exactly one place
and has no backup — don't delete it.

In [ ]:
adata = loaders.load_challenge_split(cfg, split="all", dev=True)
print(adata)
adata.obs[PERT_COL].value_counts().head()

## 2. Gene ID space

The 2025 file carries **both** ID spaces: `var_names` holds HGNC symbols and
`var['gene_id']` holds Ensembl IDs. So the symbol→Ensembl mapping the priors need
ships inside the data file — no external mapping table is required.

`gene_index` is strict: it raises rather than silently falling back to
`var_names`, which used to yield an index matching *no* prior (every source
returning zero edges, with no error).

In [ ]:
gi = loaders.gene_index(adata, "ensembl_gene_id")
print(f"{len(gi):,} Ensembl IDs resolved")

registry = load_registry("../configs/data_sources.yaml", gi, id_type="ensembl_gene_id")
print(registry.describe())

## 3. Normalize

cell-eval refuses discrete (raw count) input: MAE on raw counts is dominated by
library-size differences. Normalize to 10k counts + log1p.

In [ ]:
if loaders.is_discrete(adata):
    adata = loaders.normalize_counts(adata)
print("discrete now?", loaders.is_discrete(adata))

## 4. Split

Holdout perturbations are carved deterministically. Only perturbations with
≥30 cells are eligible: a 9-cell perturbation gives a pseudobulk too noisy to
score, so holding it out measures sampling noise, not model skill.

In [ ]:
train_perts, holdout_perts = loaders.carve_holdout(
    adata, n_holdout=25, pert_col=PERT_COL, control_label=CONTROL,
    seed=cfg["splits"]["seed"],
)
labels = adata.obs[PERT_COL].astype(str)
train = adata[labels.isin(set(train_perts) | {CONTROL}).to_numpy()].copy()
real  = adata[labels.isin(set(holdout_perts) | {CONTROL}).to_numpy()].copy()
print(f"train {train.n_obs:,} cells / {len(train_perts)} perts")
print(f"holdout {real.n_obs:,} cells / {len(holdout_perts)} perts")
holdout_perts[:10]

## 5. Pseudobulk (inspection)

Not used by the rungs below — they work on cells so DES has dispersion — but this
is the Rung-1 view of the data and what a delta looks like.

In [ ]:
pb = pseudobulk(train, [PERT_COL], aggregate="mean", min_cells=25)
deltas = delta_vs_control(pb, pert_col=PERT_COL, control_label=CONTROL)
print(pb.shape, "->", deltas.shape)
deltas.iloc[:5, :5]

## 6. Climb the ladder

cell-eval requires pred and real to carry the **same** perturbation set, control
included, so mirror the real cell counts exactly.

In [ ]:
real_labels = real.obs[PERT_COL].astype(str)
counts = {p: int((real_labels == p).sum()) for p in holdout_perts}
n_control = int((real_labels == CONTROL).sum())

RUNGS = {
    "Rung 0  predict-control": PredictControl,
    "Rung 0b predict-mean-perturbation": PredictMeanPerturbation,
    "Rung 1  statistical backbone": StatisticalBackbone,
}

results = {}
for label, cls in RUNGS.items():
    model = cls(pert_col=PERT_COL, control_label=CONTROL).fit(train)
    pred = model.predict(counts, seed=0, include_control=n_control)
    results[label] = score(
        pred, real,
        eval_config="../configs/eval.yaml",
        challenge_config="../challenges/vcc2025/config.yaml",
        outdir=f"../runs/notebook/{label.split()[1]}",
    )
    print(f"\n--- {label} ---")
    print(summarize(results[label]))

## 7. Compare

DES ↑ · PDS ↑ · MAE ↓

In [ ]:
import pandas as pd
tbl = pd.DataFrame({k: v["challenge_metrics"] for k, v in results.items()}).T
tbl.index.name = "rung"
tbl

## 8. What the numbers say

Check these before trusting a gain:

- **Does Rung 1 beat Rung 0b?** Rung 0b (every perturbation gets the average
  training shift) is the honest floor — it already captures "cells respond to
  being perturbed at all". Beating *predict-control* means nothing.
- **Is PDS above chance?** All three rungs predict a nearly identical profile for
  every perturbation, so PDS sits near 0.5 by construction. Real PDS gain needs
  perturbation-*specific* signal, which is what the prior graph is for.
- **Did the variance-inflation guardrail fire?** Scaling predicted deltas up buys
  PDS separation without predicting biology better. `summarize()` reports it.
- **`de_nsig_counts_pred` vs `de_nsig_counts_real`** — how many DE genes the
  prediction calls versus the truth. Order-of-magnitude disagreement means the
  delta is mis-scaled even when DES looks fine.